# 📡 Notebook 4: Server-Sent Events (SSE)

SSE is a powerful upgrade to long polling - the server can send **multiple messages** over a **single connection**! It's perfect for one-way real-time updates.

## Learning Objectives

By the end of this notebook, you'll understand:
- How SSE works (chunked transfer encoding)
- The SSE message format
- When to use SSE vs other methods
- How to build a real-time dashboard with SSE

## 🤔 What is Server-Sent Events?

With long polling, after each message, you need to make a **new request**. SSE keeps the connection open!

```
Long Polling:                      SSE:
                                   
Request  ──►                       Request  ──►
Response ◄── (message 1)           ◄── message 1 (chunked)
Request  ──►                       ◄── message 2 (chunked)
Response ◄── (message 2)           ◄── message 3 (chunked)
Request  ──►                       ◄── message 4 (chunked)
Response ◄── (message 3)           ...continues...

Multiple connections!              Single connection!
```

The magic: **chunked transfer encoding** - the server doesn't have to know the response size upfront!

## 📝 SSE Message Format

SSE has a simple text-based format:

```
data: {"message": "Hello!"}

data: {"message": "Another one!"}

event: custom_event
data: {"type": "notification"}

id: 42
data: {"message": "With an ID!"}

```

Each message ends with **two newlines**. Key fields:
- `data:` - The actual message content (required)
- `event:` - Custom event type (optional)
- `id:` - Message ID for reconnection (optional)
- `retry:` - Reconnection interval in ms (optional)

## 🛠️ Let's Build It!

### Step 1: Start the Server

The next cell starts `servers/sse_server.py` for you and shuts it down when
the kernel exits. To watch its log live instead, start it yourself in a second
terminal first — the notebook leaves an already-listening port alone:

```bash
cd 04-patterns/real-time-updates/servers
python sse_server.py     # 🚀 Starting SSE Server on port 5003
```

In [ ]:
# Start the SSE server this notebook talks to.
#
# `ensure_server` is idempotent: if you already started the server by hand
# in another terminal it is left alone, otherwise it is launched as a
# background process using this notebook's own interpreter (the lab .venv)
# and shut down when the kernel exits. This is what makes the notebook
# runnable on its own -- previously the next cell just died with a raw
# ConnectionError if you had not started the server first.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "servers"))
from lab_servers import ensure_server

print(ensure_server(5003))

import requests

health = requests.get("http://localhost:5003/health", timeout=5)
health.raise_for_status()
print("health:", health.json())


### Step 2: Create the SSE Client

In [ ]:
import requests
import json
import threading
import time
from datetime import datetime

class SSEClient:
    """
    A client for consuming Server-Sent Events.
    """
    
    def __init__(self, url: str):
        self.url = url
        self.running = False
        self.last_id = None
        self.messages = []
    
    def parse_sse(self, line: str):
        """
        Parse a single SSE line.
        """
        if line.startswith("data:"):
            return ("data", line[5:].strip())
        elif line.startswith("event:"):
            return ("event", line[6:].strip())
        elif line.startswith("id:"):
            return ("id", line[3:].strip())
        return None
    
    def listen(self, duration: int = None, callback=None, heartbeat: float = 1.0):
        """
        Listen for SSE events.

        Args:
            duration: How long to listen (None = forever)
            callback: Function to call for each message
            heartbeat: How often to ask the server to send a keep-alive.

        Why `heartbeat` matters here: the loop below can only notice that
        `duration` has elapsed when a line arrives. On a quiet stream with the
        server's 15s default we would sit here for 15 seconds no matter what
        `duration` said. Asking for a 1s heartbeat bounds that overshoot --
        and it is what production SSE endpoints do anyway, so that proxies
        don't kill the idle connection.
        """
        self.running = True
        start_time = time.time()
        
        headers = {
            "Accept": "text/event-stream",
            "Cache-Control": "no-cache",
        }
        
        # Include last event ID for reconnection
        if self.last_id:
            headers["Last-Event-ID"] = str(self.last_id)
        
        try:
            # stream=True enables chunked reading. The read timeout is a
            # backstop: if even the heartbeat stops arriving, the stream is
            # dead and we should raise rather than hang the notebook.
            response = requests.get(
                self.url,
                headers=headers,
                params={"heartbeat": heartbeat},
                stream=True,
                timeout=(5, heartbeat * 5 + 5),
            )
            
            current_event = None
            current_data = None
            current_id = None
            
            for line in response.iter_lines(decode_unicode=True):
                # Check if we should stop
                if not self.running:
                    break
                if duration and (time.time() - start_time) > duration:
                    break
                
                if line:
                    parsed = self.parse_sse(line)
                    if parsed:
                        field, value = parsed
                        if field == "data":
                            current_data = value
                        elif field == "event":
                            current_event = value
                        elif field == "id":
                            current_id = value
                            self.last_id = value
                
                else:
                    # Empty line = end of message
                    if current_data:
                        try:
                            data = json.loads(current_data)
                        except json.JSONDecodeError:
                            data = current_data
                        
                        message = {
                            "event": current_event or "message",
                            "data": data,
                            "id": current_id,
                            "received_at": datetime.now().isoformat()
                        }
                        
                        self.messages.append(message)
                        
                        if callback:
                            callback(message)
                        
                        # Reset for next message
                        current_event = None
                        current_data = None
                        current_id = None
                        
        except Exception as e:
            print(f"❌ SSE connection error: {e}")
        finally:
            self.running = False
            try:
                response.close()   # release the socket; the server sees the
            except Exception:      # disconnect and drops our subscriber queue
                pass
    
    def stop(self):
        """Stop listening."""
        self.running = False

def send_message(user: str, text: str):
    """Send a message via the HTTP API."""
    try:
        response = requests.post(
            "http://localhost:5003/send",
            json={"user": user, "text": text},
            timeout=5
        )
        return response.status_code == 201
    except:
        return False

print("✅ SSE client created!")

## 🧪 Experiment: See SSE in Action!

In [ ]:
# Let's see SSE in action with a simple listener

def on_message(message):
    """Callback for each received message."""
    event = message['event']
    data = message['data']
    
    if event == "connection":
        print(f"🔗 Connected: {data.get('message')}")
    elif event == "heartbeat":
        print(f"💓 Heartbeat received")
    elif event == "message":
        print(f"📬 Message: {data.get('user')}: {data.get('text')}")
    else:
        print(f"📨 {event}: {data}")

# Create client
client = SSEClient("http://localhost:5003/events")

# Send some messages while listening
def send_test_messages():
    time.sleep(1)
    print("\n📤 Sending test messages...")
    send_message("Alice", "Hello SSE!")
    time.sleep(1)
    send_message("Bob", "This is real-time!")
    time.sleep(1)
    send_message("Charlie", "Amazing!")

# Start message sender
sender = threading.Thread(target=send_test_messages)
sender.start()

# Listen for 5 seconds. The 2s heartbeat is what lets the loop notice that
# `duration` elapsed once the messages stop -- with the server's 15s default
# this cell would sit here for 15 seconds regardless of what `duration` says.
print("🔄 Listening for SSE events...\n")
started = time.time()
client.listen(duration=5, callback=on_message, heartbeat=2)
elapsed = time.time() - started

sender.join()

chat = [m for m in client.messages if m["event"] == "message"]
print(f"\n✅ Done in {elapsed:.1f}s. {len(chat)} message(s) over ONE connection.")

assert len(chat) == 3, f"expected all 3 messages on one stream, got {len(chat)}"
assert all(m["id"] for m in chat), "every message must carry an id -- that is what Last-Event-ID resumes from"
assert elapsed < 9, f"listen(duration=5) ran for {elapsed:.1f}s -- duration is not being honoured"
print("   Every message carried an `id:` -- notebook's reconnection demo needs those.")

## 📊 SSE vs Long Polling: Burst Messages

Let's see how SSE handles burst messages better than long polling:

In [ ]:
# Demonstrate SSE handling burst messages efficiently.
#
# Notebook 3 measured the same burst over long polling and counted one HTTP
# round trip per message. Here we measure END-TO-END latency (publish ->
# client callback) and the number of connections it took. That is the fair
# comparison: same burst, same loopback, different transport.

def burst_test(n=5, gap=0.1):
    print("📊 Burst Message Test")
    print("="*50)

    burst_client = SSEClient("http://localhost:5003/events")
    sent_at = {}
    received = []          # (text, latency_seconds)

    def track_message(message):
        if message['event'] == 'message':
            text = message['data'].get('text')
            if text in sent_at:
                latency = time.time() - sent_at[text]
                received.append((text, latency))
                print(f"📬 Received: {text} ({latency*1000:.1f}ms after publish)")

    def send_burst():
        time.sleep(1)  # wait for the stream to be established
        print(f"\n📤 Sending {n} messages with {int(gap*1000)}ms gaps...\n")
        for i in range(n):
            text = f"Message {i+1}"
            sent_at[text] = time.time()
            send_message("Burst", text)
            time.sleep(gap)

    sender = threading.Thread(target=send_burst)
    sender.start()
    burst_client.listen(duration=4, callback=track_message, heartbeat=1)
    sender.join()

    latencies = [lat for _, lat in received]
    print(f"\n📊 Results:")
    print(f"   Messages sent:      {n}")
    print(f"   Messages received:  {len(received)}")
    print(f"   HTTP connections:   1")
    print(f"   Median latency:     {statistics.median(latencies)*1000:.1f}ms")
    print(f"   Worst latency:      {max(latencies)*1000:.1f}ms")
    print(f"\n💡 {n} messages, ONE connection. Long polling needed one request per")
    print("   message for the same burst (notebook 3, case A) -- and every one of")
    print("   those requests pays a full network round trip in production.")

    assert len(received) == n, f"only {len(received)}/{n} burst messages arrived"
    assert max(latencies) < 1.0, (
        f"worst SSE latency was {max(latencies)*1000:.0f}ms on loopback -- the "
        f"stream is buffering instead of pushing"
    )
    return received

import statistics
burst_test()

## 🔄 Automatic Reconnection

SSE has built-in reconnection support using the `id` and `Last-Event-ID` header:

In [ ]:
# Demonstrate SSE reconnection concept

print("🔄 SSE Reconnection Concept")
print("="*50)
print("""
When the connection drops, browsers automatically reconnect!

How it works:

1. Server sends messages with IDs:
   id: 1
   data: {"message": "First"}
   
   id: 2
   data: {"message": "Second"}

2. Connection drops...

3. Client reconnects with header:
   Last-Event-ID: 2

4. Server sends only messages AFTER ID 2!

This means: NO MISSED MESSAGES! 🎉
""")

print("Our client stores the last ID:")
print(f"   client.last_id = {client.last_id}")
print("\nOn reconnection, it would send: Last-Event-ID: " + str(client.last_id))

## 🌐 Browser Support

Modern browsers have built-in SSE support via the `EventSource` API:

```javascript
// Browser JavaScript - it's this simple!
const eventSource = new EventSource('/events');

eventSource.onmessage = (event) => {
    console.log('Received:', event.data);
};

eventSource.onerror = () => {
    console.log('Connection lost, auto-reconnecting...');
};

// To close:
// eventSource.close();
```

The browser handles:
- Connection management
- Automatic reconnection
- Last-Event-ID tracking

## ✅ Advantages of SSE

1. **Single connection** - Multiple messages, one connection
2. **Built into browsers** - `EventSource` API
3. **Automatic reconnection** - Browser handles it
4. **Message IDs** - No missed messages
5. **Still HTTP** - Works with existing infrastructure
6. **Text-based** - Easy to debug

## ❌ Disadvantages

1. **One-way only** - Server → Client (no client → server)
2. **Proxy issues** - Some proxies buffer responses
3. **Browser limits** - ~6 connections per domain
4. **Text only** - Binary data needs encoding
5. **Monitoring** - Long-lived requests look odd in metrics

In [ ]:
def check_sse_stats():
    try:
        response = requests.get("http://localhost:5003/stats")
        stats = response.json()
        
        print("📊 SSE Server Statistics")
        print("="*40)
        print(f"   Connected clients: {stats['connected_clients']}")
        print(f"   Messages sent:     {stats['messages_sent']}")
        print(f"   Buffer size:       {stats['buffer_size']}/{stats['buffer_max_size']}")
    except:
        print("❌ Could not fetch stats")

check_sse_stats()

## 🔄 Deep Dive: Last-Event-ID and Reconnection

One of SSE's most powerful features is **automatic message recovery**. Here's how it works:

### The Problem: What happens when a connection drops?

```
Client connected ──────> receives msg 1, 2, 3
        |
   [Network drops]
        |
   (messages 4, 5 sent while disconnected)
        |
Client reconnects ────> HOW does it get messages 4, 5?
```

### The Solution: Message IDs and Last-Event-ID Header

1. **Server assigns IDs** to each message: `id: 123`
2. **Client tracks** the last received ID
3. **On reconnect**, client sends header: `Last-Event-ID: 123`
4. **Server replays** messages with ID > 123

This is why our updated server maintains a **message buffer**!

In [ ]:
# Let's demonstrate Last-Event-ID reconnection!

import json
import requests
import time

def demonstrate_reconnection():
    """
    This demonstrates how Last-Event-ID works:
    1. Send some messages (which get buffered)
    2. Simulate a client disconnecting  
    3. Send messages while client is "offline"
    4. Reconnect WITH Last-Event-ID header
    5. See the missed messages get replayed!

    Crucially we do not just *print* the replay -- we collect the ids the
    server sends back and check they are exactly the ids we missed. A replay
    that silently returned nothing (or everything) would print something
    plausible-looking either way.
    """
    print("🎯 Last-Event-ID Demonstration")
    print("="*50)
    
    # Step 1: Check the current message buffer
    print("\n📦 Step 1: Check current message buffer")
    buffer_data = requests.get("http://localhost:5003/buffer", timeout=5).json()
    print(f"   Messages in buffer: {buffer_data['count']}")
    if buffer_data['messages']:
        last_msg = buffer_data['messages'][-1]
        print(f"   Last message ID: {last_msg['id']}")
        print(f"   Last message: '{last_msg['text']}'")
    
    # Step 2: Send some new messages  
    print("\n📨 Step 2: Sending 3 messages to create a baseline...")
    for i in range(3):
        response = requests.post(
            "http://localhost:5003/send",
            json={"user": "demo", "text": f"Message {i+1} - online"},
            timeout=5,
        )
        msg = response.json()
        print(f"   Sent: '{msg['text']}' (ID: {msg['id']})")
        time.sleep(0.3)
    
    # Record the last ID we "saw"
    last_seen_id = msg['id']
    print(f"\n💾 Client remembers: Last-Event-ID = {last_seen_id}")
    
    # Step 3: Simulate being offline - send messages without receiving
    print("\n📴 Step 3: Client goes OFFLINE (simulated disconnect)")
    print("   Sending 3 more messages while client is 'offline'...")
    
    missed_ids = []
    for i in range(3):
        response = requests.post(
            "http://localhost:5003/send",
            json={"user": "demo", "text": f"Message {i+1} - MISSED (sent while offline)"},
            timeout=5,
        )
        msg = response.json()
        missed_ids.append(msg["id"])
        print(f"   Server sent: '{msg['text']}' (ID: {msg['id']})")
        time.sleep(0.3)
    
    # Step 4: Reconnect with Last-Event-ID
    print(f"\n📱 Step 4: Client RECONNECTS with Last-Event-ID: {last_seen_id}")
    print("   Watch - the server should replay missed messages!\n")
    
    headers = {
        "Accept": "text/event-stream",
        "Last-Event-ID": str(last_seen_id),   # this is what browsers send
    }
    
    # Ask for a fast heartbeat so the stream produces a line quickly once the
    # replay is done and we can stop reading without waiting 15s.
    response = requests.get(
        "http://localhost:5003/events",
        headers=headers,
        params={"heartbeat": 1},
        stream=True,
        timeout=(5, 10),
    )
    
    print("   📥 Messages received on reconnect:")
    replayed_ids, saw_connection, current_id = [], False, None
    try:
        for line in response.iter_lines(decode_unicode=True):
            if line.startswith("id:"):
                current_id = int(line[3:].strip())
            elif line.startswith("data:"):
                payload = json.loads(line[5:].strip())
                if payload.get("type") == "connected":
                    saw_connection = True
                    print(f"      [connection] recovered_from={payload['recovered_from']}")
                elif payload.get("type") == "heartbeat":
                    break          # replay is over, stream is idle
                else:
                    replayed_ids.append(current_id)
                    print(f"      id={current_id} {payload['text']}")
    finally:
        response.close()
    
    print("\n" + "="*50)
    print(f"✅ Replayed ids: {replayed_ids}")
    print(f"   Ids we missed: {missed_ids}")

    # The assertions that make this a demonstration rather than a story:
    assert saw_connection, "server never sent the `connection` event"
    assert replayed_ids == missed_ids, (
        f"replay is wrong: got {replayed_ids}, expected exactly the missed "
        f"ids {missed_ids} -- no gaps (lost messages) and no extras (the "
        f"client would be shown things it already had)"
    )
    print("   Exactly the missed messages -- no gaps, no re-delivery of what we had.")

# Run the demonstration
demonstrate_reconnection()

### 🧠 Understanding the Flow

Here's what happened in the demo above:

```
Timeline:
─────────────────────────────────────────────────────────────────
 Online Phase          │ Offline Phase         │ Reconnection
─────────────────────────────────────────────────────────────────
 Msg 1 ✓ received      │                       │
 Msg 2 ✓ received      │                       │
 Msg 3 ✓ received      │                       │
 (last_seen_id = 3)    │                       │
                       │ Msg 4 → buffer only   │
                       │ Msg 5 → buffer only   │ 
                       │ Msg 6 → buffer only   │
                       │                       │ Connect with Last-Event-ID: 3
                       │                       │ Server replays: 4, 5, 6 ✓
─────────────────────────────────────────────────────────────────
```

### Key Points:

1. **Buffer is finite** — Our server keeps the last 100 messages. If you're offline too long, old messages are lost, and the server has no way to tell you so. Pair the stream with a snapshot endpoint (see notebook 7) for outages longer than the buffer.
2. **Browser handles this** — With `EventSource`, you don't write any reconnection code.
3. **Server must implement it** — Not all SSE servers support Last-Event-ID!
4. **Replay is at-least-once, not exactly-once** — our server registers the new subscriber queue *before* it replays the buffer, on purpose. A message published during the replay therefore arrives twice: once from the buffer, once live. That is the right trade (a gap loses data; a duplicate does not), but it means **the client must dedupe on `id`**. Doing it the other way round — replay first, subscribe after — closes the duplicate window and opens a gap instead.
5. **`retry:`** — our server emits `retry: 2000` on connect, which is how you tell `EventSource` how long to wait before reconnecting. Without it the browser picks its own default (~3s).

In [ ]:
# View the current message buffer (debug endpoint)

def view_message_buffer():
    """
    Our server has a /buffer endpoint to see what's stored.
    This is what the server uses to replay on reconnection.
    """
    print("📦 Current Message Buffer")
    print("="*50)
    
    data = requests.get("http://localhost:5003/buffer", timeout=5).json()
    stats = requests.get("http://localhost:5003/stats", timeout=5).json()
    
    # Read the capacity from the server rather than repeating "100" in prose --
    # a hardcoded number here is a lie waiting to happen.
    print(f"Total messages buffered: {data['count']}")
    print(f"Buffer capacity: {stats['buffer_max_size']} messages")
    print()
    
    if data['messages']:
        print("Last 5 messages in buffer:")
        for msg in data['messages'][-5:]:
            print(f"  ID {msg['id']:3d}: [{msg['user']}] {msg['text']}")
    else:
        print("Buffer is empty")
    
    print(f"\n📊 Server Stats:")
    print(f"   Connected clients: {stats['connected_clients']}")
    print(f"   Total messages sent: {stats['messages_sent']}")
    print(f"   Reconnect hint sent to EventSource: retry: {stats['retry_ms']}ms")
    
    assert data['count'] <= stats['buffer_max_size'], "buffer grew past its own maximum"
    # Nothing should still be attached: every listen() above closed its stream.
    assert stats['connected_clients'] == 0, (
        f"{stats['connected_clients']} SSE stream(s) still open -- a demo leaked "
        f"a connection, which on a real server means a leaked queue too"
    )

view_message_buffer()

## 🚧 Backpressure: What If the Client Can't Keep Up?

SSE looks like a one-way firehose from server to client — but **the client's TCP receive window can fill up** if the browser tab is throttled, the laptop wakes from sleep, or the network is slow. When that happens, the server's `await queue.put(msg)` in our `sse_server.py` keeps succeeding (the `asyncio.Queue` has no bound), and memory grows until something crashes.

```
Fast producer                                    Slow consumer
   📡 send ─► [queue: 5] ─► [queue: 500] ─► [queue: 5,000] ─► 💥
```

Three practical fixes — in order of increasing sophistication:

| Fix | How | Trade-off |
|-----|-----|-----------|
| **Bound the queue** | `asyncio.Queue(maxsize=1000)` — `put()` blocks when full | Back-pressures *all* subscribers (simplest, often fine) |
| **Drop on full** | `queue.put_nowait(msg)` inside `try/except QueueFull` — drop for that one slow client | Fast clients stay fast; slow clients miss events (pair with `Last-Event-ID` replay) |
| **Disconnect slow client** | If the queue has been full for N seconds, close the connection and let the client reconnect | Forces the client into the buffered replay path — it catches up via `Last-Event-ID` |

Our lab server is intentionally unbounded for teaching clarity — in production you'd combine option 2 (drop+log) with option 3 (disconnect after sustained fullness) and lean on the `Last-Event-ID` replay we built earlier so the client recovers transparently. This is the **"transport ≠ durability"** lesson again: the wire-level push doesn't give you reliability; a bounded buffer + replay strategy does.


## 🎯 When to Use SSE

SSE is perfect when:

| Use Case | Why SSE Works |
|----------|---------------|
| Live dashboards | One-way updates, high frequency |
| Stock tickers | Constant updates, read-only |
| AI chat responses | Streaming tokens as they're generated |
| Social feeds | New posts pushed to users |
| Notifications | Server pushes alerts to clients |

### Don't use SSE when:

- You need **bidirectional** communication (use WebSocket)
- You need **binary data** efficiently (use WebSocket)
- You're behind problematic **proxies** (test first!)

## 🔧 Real-World: AI Chat Streaming

A very popular use case for SSE today is AI chat applications (like ChatGPT) that stream tokens as they're generated:

In [ ]:
# Simulate AI token streaming

def simulate_ai_streaming():
    """
    Simulate how AI chat apps stream tokens via SSE.
    """
    print("🤖 Simulating AI Response Streaming")
    print("="*50)
    print("\nAI response streaming word by word:\n")
    
    # Simulated AI response
    response_text = "Hello! I'm an AI assistant. I can help you with programming, answer questions, and have conversations. How can I assist you today?"
    words = response_text.split()
    
    print("User: What can you do?")
    print("\nAI: ", end="", flush=True)
    
    # Stream each word (simulating token-by-token generation)
    for word in words:
        print(word + " ", end="", flush=True)
        time.sleep(0.1)  # Simulate generation time
    
    print("\n\n" + "="*50)
    print("\n💡 In real AI apps, each word is sent as an SSE event:")
    print("")
    print('   event: token')
    print('   data: {"text": "Hello!"}')
    print('   ')
    print('   event: token')
    print('   data: {"text": "I\'m"}')
    print('   ')
    print('   event: done')
    print('   data: {"finished": true}')

simulate_ai_streaming()

## 🧪 Quick Quiz

1. **What HTTP feature makes SSE possible?**

2. **You're building a live sports score dashboard. Would SSE be a good choice?**

3. **A user is chatting with friends. Would you use SSE or WebSocket?**

In [ ]:
# Quiz answers

print("📝 Quiz Answers")
print("="*50)
print("")
print("1. CHUNKED TRANSFER ENCODING!")
print("   The server doesn't send Content-Length, so it can")
print("   keep sending data chunks indefinitely.")
print("")
print("2. YES! SSE is perfect for live dashboards.")
print("   Users only receive updates, they don't send data.")
print("")
print("3. WEBSOCKET! Chat requires bidirectional communication.")
print("   Users need to both send AND receive messages.")
print("   SSE is one-way only (server → client).")

## 📊 Comparison Table

| Feature | Simple Polling | Long Polling | SSE |
|---------|---------------|--------------|-----|
| Latency | High (interval) | Low | Very Low |
| Connection | New each time | New each response | Persistent |
| Direction | Request-Response | Request-Response | Server → Client |
| Burst handling | ✅ Consistent | ⚠️ Adds latency | ✅ Efficient |
| Browser support | ✅ Native | ✅ Native | ✅ EventSource API |
| Reconnection | Manual | Manual | Automatic |
| Complexity | Very Low | Low | Medium |

## 📚 Summary

### What We Learned:

1. **SSE** = Server sends multiple messages over ONE connection
2. Uses **chunked transfer encoding** (no Content-Length)
3. Simple **text format** with data, event, id fields
4. **Browser built-in** support via `EventSource`
5. **Automatic reconnection** with Last-Event-ID
6. **One-way only** - server to client

### Interview Tips:

> "SSE is perfect when I need server-to-client updates efficiently. It's more efficient than long polling because all messages come over one connection, and browsers handle reconnection automatically. If I needed bidirectional communication, I'd use WebSocket instead."

### Next Up: WebSockets

In the next notebook, we'll explore **WebSockets** - the full-duplex solution for bidirectional real-time communication!